# Mosaic & merge

Combine several rasters into one:

- **`merge_rasters(src, dst)`** — mosaic many (overlapping or adjacent) rasters into a single
  raster covering their union.
- **`stack_bands(files, path=...)`** — stack several single-band rasters into one multi-band
  raster (e.g. assembling per-band Sentinel files into one image).

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # never trigger an interactive backend

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-ops-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset
from pyramids.dataset.merge import merge_rasters, stack_bands

tiles = [str(DATA / 'crop_aligned_folder' / f'{i}.tif') for i in range(3)]
[Dataset.read_file(t).shape for t in tiles]

2026-06-08 23:01:55 | INFO | pyramids.base.config | Logging is configured.


[(1, 24, 29), (1, 24, 29), (1, 24, 29)]

## Mosaic — `merge_rasters`

Writes one raster covering the inputs' combined extent. `method` controls how overlaps resolve (`'last'`, `'first'`, …).

In [3]:
mosaic = WORK / 'mosaic.tif'
merge_rasters(tiles, str(mosaic))
Dataset.read_file(str(mosaic)).shape

(1, 24, 29)

## Stack bands — `stack_bands`

Combine single-band rasters into one multi-band raster (order = input order).

In [4]:
stacked = WORK / 'stacked.tif'
stack_bands(tiles, path=str(stacked), band_names=['t0', 't1', 't2'])
out = Dataset.read_file(str(stacked))
out.band_count, out.band_names

(3, ['t0', 't1', 't2'])

## Notes

- `merge_rasters` takes `no_data_value=` and `method=` to control fill and overlap handling.
- `stack_bands` can `align=True` to snap mismatched grids before stacking.
- See also: [Reproject / resample / align](reproject-resample-align.ipynb).